# 15.2 GAN, Diffusion, 그리고 세 원리 비교 — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml1/chapter15_2_gan_diffusion.ipynb)

책 본문: [Chapter 15](https://smhanlab.com/book-ml/kor/ml1/chapter15.html)

1차원 장난감 GAN으로 "내쉬 균형"(\(D^*(0)=0.5\))과 교대 최적화 궤적(책
그림 1)을 직접 계산해봅니다. 그다음 GAN의 판별자/생성자 손실을 균형점에서
구해 보고, Diffusion의 "노이즈 예측 = 가장 흔한 MSE 회귀" 아이디어를
1차원 장난감으로 확인합니다.


## 1. 장난감 GAN 세팅: \(D^*\)의 닫힌 형태와 게임 값

데이터 \(x \sim \mathcal{N}(0,1)\), 생성자 \(p_G = \mathcal{N}(0, s^2)\) —
표준편차 \(s\) 하나만 자유(평균은 0으로 고정, 책 15.2절).
\(G\)를 고정하고 \(D\)를 완전히 최적화하면 닫힌 형태
\(D^*(x) = p_{\text{data}}/(p_{\text{data}}+p_G)\)가 나오고,
이 \(D^*\)를 목적함수에 대입하면 게임 값이
\(V(D^*,G) = 2\,\mathrm{JSD}(p_{\text{data}} \| p_G) - \log 4\)
(GAN 원논문 Theorem 7)로 닫힌 형태가 됩니다.


In [1]:
import numpy as np
from scipy.stats import norm

trapz = np.trapezoid if hasattr(np, "trapezoid") else np.trapz  # numpy 1.x/2.x 호환

X = np.linspace(-8, 8, 4001)
P_DATA = norm.pdf(X)  # 데이터 분포 N(0,1)

def p_G(s):
    return norm.pdf(X, 0.0, s)  # 생성자 분포 N(0, s^2)

def D_star_at_0(s):
    """판별자 D*가 데이터 중심 x=0을 '진짜'로 판정하는 확률"""
    i = 2000  # X가 대칭이라 중앙 인덱스가 x=0
    return P_DATA[i] / (P_DATA[i] + p_G(s)[i])

def jsd(s):
    p, q = P_DATA, p_G(s)
    m = 0.5 * (p + q)
    return 0.5 * trapz(p * np.log(p / m), X) + 0.5 * trapz(q * np.log(q / m), X)

def game_value(s):
    return 2.0 * jsd(s) - np.log(4.0)

print(f"초기(s=2.0): V(D*,G) = {game_value(2.0):.3f}, D*(0) = {D_star_at_0(2.0):.3f}")
print(f"균형(s=1.0): V(D*,G) = {game_value(1.0):.3f} (=-log 4 = {-np.log(4):.3f}), D*(0) = {D_star_at_0(1.0):.3f}")


초기(s=2.0): V(D*,G) = -1.201, D*(0) = 0.667
균형(s=1.0): V(D*,G) = -1.386 (=-log 4 = -1.386), D*(0) = 0.500


## 2. 교대 최적화: \(s_0=2.0\), \(\eta=0.15\)로 40스텝

책 15.2절의 학습 루프: (1) \(G\) 고정 → \(D^*\)로 점프(장난감이라
닫힌 형태), (2) \(D^*\) 고정 → \(s\)에 그라디언트 스텝 한 스텝
(수치 미분으로 구함). 4.4절 EM의 좌표별 최적화와 같은 뼈대지만,
"잠재변수" 자리에 "네트워크"가 들어간 형태입니다.


In [2]:
s, eta, n_steps = 2.0, 0.15, 40
traj = []
for step in range(n_steps + 1):
    traj.append((step, s, game_value(s), D_star_at_0(s)))
    # (2) D*를 고정하고 s를 그라디언트 스텝으로 한 스텝 갱신
    h = 1e-5
    grad = (game_value(s + h) - game_value(s - h)) / (2 * h)
    s -= eta * grad

for step, s, v, d0 in [traj[0], traj[10], traj[20], traj[30], traj[40]]:
    print(f"스텝={step:>2}  s={s:.3f}  V(D*,G)={v:.3f}  D*(0)={d0:.3f}")
print(f"\n한계: s* = 1.0, V -> -log 4 ≈ -1.386, D*(0) -> 0.5")


스텝= 0  s=2.000  V(D*,G)=-1.201  D*(0)=0.667
스텝=10  s=1.665  V(D*,G)=-1.276  D*(0)=0.625
스텝=20  s=1.339  V(D*,G)=-1.346  D*(0)=0.572
스텝=30  s=1.113  V(D*,G)=-1.381  D*(0)=0.527
스텝=40  s=1.026  V(D*,G)=-1.386  D*(0)=0.506

한계: s* = 1.0, V -> -log 4 ≈ -1.386, D*(0) -> 0.5


## 3. 책 그림 1: 게임 값과 판별자 출력의 궤적

왼쪽: 게임 값 \(V(D^*,G)\)가 하한 \(-\log 4 \approx -1.386\)으로
**단조 감소**(0으로 가지 않는다 — (iii)). 오른쪽: \(D^*(0)\)가 0.5로
수렴 — 균형을 향해 갈수록 판별자가 진짜/가짜를 구별할 수 없어짐((ii)).


In [3]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Noto Sans CJK KR"
plt.rcParams["axes.unicode_minus"] = False

steps   = [t[0] for t in traj]
v_vals  = [t[2] for t in traj]
d0_vals = [t[3] for t in traj]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

ax = axes[0]
ax.plot(steps, v_vals, "o-", color="#1f77b4", ms=3, lw=1.2)
ax.axhline(-np.log(4), color="gray", ls="--", lw=1.0)
ax.text(2, -np.log(4) + 0.004, r"$V_{min}=-\log 4\approx-1.386$ (lower bound)", fontsize=9, color="gray")
ax.set_xlabel("Alternating optimization step"); ax.set_ylabel(r"$V(D^*, G)$")
ax.set_title("(a) Game value: monotonically decreases toward the lower bound -log 4 (s → 1)")
ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(steps, d0_vals, "o-", color="#ff7f0e", ms=3, lw=1.2)
ax.axhline(0.5, color="gray", ls="--", lw=1.0)
ax.text(2, 0.506, r"$D^*(0)=0.5$ (perfectly indistinguishable = Nash equilibrium)", fontsize=9, color="gray")
ax.set_xlabel("Alternating optimization step"); ax.set_ylabel(r"$D^*(0)$")
ax.set_title("(b) Discriminator output at the data center: converges to 0.5")
ax.grid(alpha=0.3)

fig.tight_layout()
fig.savefig("ch15_3_gan_dynamics.svg", bbox_inches="tight")
print("saved: ch15_3_gan_dynamics.svg  (-> kor/src/images/ 에 복사해 본문에 삽입)")


saved: ch15_3_gan_dynamics.svg  (-> kor/src/images/ 에 복사해 본문에 삽입)


## 4. 판별자/생성자 손실과 내쉬 균형 (책 연습문제 2)

책 15.2절의 손실 함수를 그대로 구현합니다. 내쉬 균형에서는
\(D(x) = D(G(z)) = 0.5\) — 이때 두 손실 값이 0이 **아니라는**
점을 기억하라 ("loss가 0에 가까워지면 성공"은 틀린 읽기, 책 실수 2).


In [4]:
import math

def discriminator_loss(D_real, D_fake):
    return -(math.log(D_real) + math.log(1 - D_fake))  # 판별자가 최소화할 손실

def generator_loss(D_fake):
    return -math.log(D_fake)  # 생성자가 최소화할 손실 (D_fake를 1에 가깝게)

print(f"D 잘 구별할 때 (D_real=0.9, D_fake=0.1): D_loss = {discriminator_loss(0.9, 0.1):.4f}")
print(f"G 잘 속일 때 (D_fake=0.9):               G_loss = {generator_loss(0.9):.4f}")
print()
print("내쉬 균형 (D = 0.5)에서:")
print(f"  D_loss = {discriminator_loss(0.5, 0.5):.4f}  (= log 4 ≈ 1.386)")
print(f"  G_loss = {generator_loss(0.5):.4f}  (= log 2 ≈ 0.693)")
print("  -> 균형은 '손실 0'이 아니라 '손실이 더 이상 안 줄어드는 점'")


D 잘 구별할 때 (D_real=0.9, D_fake=0.1): D_loss = 0.2107
G 잘 속일 때 (D_fake=0.9):               G_loss = 0.1054

내쉬 균형 (D = 0.5)에서:
  D_loss = 1.3863  (= log 4 ≈ 1.386)
  G_loss = 0.6931  (= log 2 ≈ 0.693)
  -> 균형은 '손실 0'이 아니라 '손실이 더 이상 안 줄어드는 점'


## 5. Diffusion 장난감: 노이즈 예측은 가장 흔한 MSE 회귀

정방향 과정: \(x_t = \alpha_t x_0 + \sigma_t \epsilon\)
(선형 스케줄 \(\bar\alpha_t = 1 - t/T\), \(\epsilon \sim
\mathcal{N}(0,1)\)) — 학습 대상이 아닌 고정 절차. 학습 대상은
\(x_t\)에 더해진 노이즈 \(\epsilon\)을 예측하는 네트워크
\(\epsilon_\theta(x_t, t)\) 하나뿐이고, 손실은 MSE — 2장의
선형회귀와 **동일한 기법**입니다.

\((\epsilon, x_t)\)가 결합 가우시안이므로 최선 선형 예측기는
\(\mathbb{E}[\epsilon \mid x_t] = \mathrm{Cov}(\epsilon, x_t)
\cdot x_t / \mathrm{Var}(x_t) = \sigma_t\, x_t\) — 기울기가
정확히 \(\sigma_t\)가 됩니다. 최소제곱으로 실제로 맞혀 보고,
이 예측이 스코어 함수 \(\nabla_x \log p_t(x)\)에 비례함을
확인합니다(책 본문, Song & Ermon 2019).


In [5]:
rng = np.random.default_rng(7)
T, N = 1000, 50_000

def schedule(t):
    alpha_bar = 1.0 - t / T          # 선형 스케줄
    alpha = np.sqrt(alpha_bar)
    sigma = np.sqrt(1.0 - alpha_bar)
    return alpha, sigma

print(f"{'t':>5} | {'맞춘 기울기':>12} | {'이론 σ_t':>10} | {'MSE':>8} | {'이론 잔차 α_t²':>12}")
print("-" * 58)
for t in [10, 250, 500, 750, 990]:
    alpha, sigma = schedule(t)
    x0 = rng.standard_normal(N)
    eps = rng.standard_normal(N)
    x_t = alpha * x0 + sigma * eps          # 정방향 (닫힌 형태)
    # 최소제곱: ε_θ(x_t, t) = w · x_t
    w = np.sum(x_t * eps) / np.sum(x_t ** 2)
    mse = np.mean((eps - w * x_t) ** 2)
    print(f"{t:>5} | {w:>12.4f} | {sigma:>10.4f} | {mse:>8.4f} | {alpha**2:>12.4f}")
print()
print("맞춘 기울기 ≈ 이론 최적 기울기 σ_t — '노이즈 예측'은 답이 알려진")
print("선형회귀 문제일 뿐. 그래서 Diffusion이 '가장 흔한 최적화'로 안정하다.")


    t |       맞춘 기울기 |     이론 σ_t |      MSE |   이론 잔차 α_t²
----------------------------------------------------------
   10 |       0.1009 |     0.1000 |   0.9896 |       0.9900
  250 |       0.4933 |     0.5000 |   0.7525 |       0.7500
  500 |       0.7054 |     0.7071 |   0.4982 |       0.5000
  750 |       0.8654 |     0.8660 |   0.2497 |       0.2500
  990 |       0.9950 |     0.9950 |   0.0100 |       0.0100

맞춘 기울기 ≈ 이론 최적 기울기 σ_t — '노이즈 예측'은 답이 알려진
선형회귀 문제일 뿐. 그래서 Diffusion이 '가장 흔한 최적화'로 안정하다.
